# 训练数据准备与启动前检查

In [1]:
import json, os, sys, io, tempfile
from pathlib import Path
from IPython.display import display, SVG
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'documentation/assets').is_dir())
from openmolspec.dataset import MolecularSpectra, list_datasets
print('Registered datasets:', [c['name'] for c in list_datasets()])

Registered datasets: ['opendatalab_nmr', 'nmrspec_test', 'patent_test_10k', 'nmrexp_exp_10k', 'nmrgym_test', 'openspec_test', 'massspecgym_test', 'massspecgym_1_5']


分别取 OpenDataLab / WanJuan 原训练集、验证集的两个真实记录，按可读大小及核种/分子式相容性选择。这些是软件检查样本，不是性能基准。不训练模型、不更新优化器。

In [2]:
data = MolecularSpectra.from_jsonl(splits={s: root / f'documentation/assets/sdk-nmr-{s}-examples.jsonl' for s in ('train', 'val')}, name='documented_opendatalab_train_val', domain='experimental')
for role, view in data.get_split().items():
    print(role, [row['smiles'] for row in view])

train ['C#CCCc1ccccc1', 'CC=C=Cc1ccccc1']
val ['c1c2c(cc3c1CC3)CC2', 'C=C1Cc2ccccc2C1']


In [3]:
from openmolspec.training import prepare_nmr_training
from openmolspec.model import NMRTransformer
prepared_tmp = tempfile.TemporaryDirectory(prefix='openmolspec-training-tutorial-')
prepared = prepare_nmr_training(data['train'], data['val'], Path(prepared_tmp.name) / 'data')
model = NMRTransformer(stage='align')
plan = model.training_plan(prepared, run_dir=Path(prepared_tmp.name) / 'run', epochs=1, batch_size=2, parameters={'hidden_size': 32, 'heads': 4, 'encoder_layers': 1, 'decoder_layers': 1, 'feedforward_size': 64, 'projection_size': 16, 'max_train_batches': 1, 'max_val_examples': 2})
print(json.dumps(plan.validate(), indent=2))
print(json.dumps(plan.describe(), indent=2))
assert not plan.run_dir.exists()

{
  "validated": true,
  "training_performed": false,
  "uses_test_split": false,
  "split_audit": {
    "identity": "connectivity",
    "nmr_chemistry_checked": true,
    "records": {
      "train": 2,
      "val": 2
    },
    "within_split_duplicates": {
      "train": 0,
      "val": 0
    },
    "overlaps": [
      {
        "left": "train",
        "right": "val",
        "molecules": 0
      }
    ],
    "disjoint": true,
    "scope": "supplied views only; filtered/limited views do not establish full-dataset independence"
  }
}
{
  "command": [
    "/mnt/shared-storage-user/xiejiaqing/OpenMolSpec/studio/.runtime/docs-sphinx/bin/python",
    "-m",
    "spec2struct_fork.train",
    "align",
    "--data-dir",
    "/tmp/openmolspec-training-tutorial-2pujak9h/data",
    "--run-dir",
    "/tmp/openmolspec-training-tutorial-2pujak9h/run",
    "--epochs",
    "1",
    "--batch-size",
    "2",
    "--learning-rate",
    "0.0001",
    "--seed",
    "2026",
    "--device",
    "cpu",
    "

安装训练依赖后，显式调用 model.fit(plan) 才启动训练；这里未执行。另一次实际运行已完成编码器和生成器各一步训练，并验证保存、重载与推理结果一致。下方检查保存的报告；本 notebook 不启动这次训练。旧权重只用于初始化，不恢复优化器/调度器状态。

In [4]:
roundtrip = json.loads((root / 'documentation/assets/reports/sdk_training_roundtrip_20261117.json').read_text())
assert roundtrip['completed'] and roundtrip['training_performed']
assert roundtrip['test_records'] == 0
for stage, result in roundtrip['stages'].items():
    assert result['save_reload_inference_equal']
    print(stage, 'training/save/reload passed', result['checkpoint_sha256'])
print('Scope:', roundtrip['scope'])

align training/save/reload passed 13950f0306ec2a958f52daf2d8646e7a34794f62d615365911eca1c09fbf299a
generate training/save/reload passed 7458c141273390b6c5c21c2e61f4f32a1d90aad18a7df827ed7f3206868dcee0
Scope: software integration only, not a performance experiment


[Python SDK guide](../api.md)